# 第9课 (上)：打造AI裁判 —— 模型篇 (First Principles)

**本课核心目标：**
1. **理解数据 (Data)**：机器看到的“世界”是怎样的？(34个坐标 + 5个特征)
2. **数据划分 (Split)**：为什么不能死记硬背？(Train/Test 划分)
3. **构建大脑 (Model)**：设计神经网络的结构 (MLP)。
4. **训练优化 (Optimization)**：通过 Loss 曲线观察机器是如何“改正错误”的。

In [11]:
%load_ext autoreload
%autoreload 2

# 导入工具箱
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from utils import load_data, plot_loss_curve

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 第一阶段：数据是基础 —— 机器眼中的动作
在训练之前，我们先看看 csv 文件里到底存了什么。

In [12]:
# 读取原始数据文件
df_raw = pd.read_csv('training_data.csv', encoding='gbk')

# 查看前3行数据
print("=== 数据预览 (前3行) ===")
display(df_raw.head(3))

# 查看数据维度
print(f"\n数据总行数: {df_raw.shape[0]} 行")
print(f"数据总列数: {df_raw.shape[1]} 列 (包含文件名、标签和所有特征)")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd7 in position 32: invalid continuation byte

### 数据的奥秘
我们的 AI 裁判并不是“看”图片，而是看**数字**。
每一行数据包含 **39 个特征**：
*   **34 个基础特征**：17 个骨骼点的 (x, y) 坐标。
*   **5 个高级特征**：左膝角、右膝角、左髋角、右髋角、腿身比（这些是我们帮 AI 算好的）。

**思考**：为什么我们不仅给坐标，还要专门算出“角度”给它？（提示：人类裁判看什么？）

## 第二阶段：记忆 vs 理解 —— 考试的必要性
机器如果把所有答案都背下来（Overfitting），遇到新题目就会懵圈。
所以我们必须把数据分成两部分：一部分用来**学习**，一部分用来**考试**。

In [ ]:
# 1. 加载处理好的数据 (自动去掉文件名等无关列)
X, y = load_data()

print("=== 特征检查 ===")
print(f"每条数据的特征数量: {X.shape[1]} 个")
if X.shape[1] == 39:
    print("✅ 特征数量正确 (34个坐标 + 5个角度/比例)")
else:
    print("⚠️ 特征数量异常，请检查数据！")

# === 【活动 1：数据划分】 ===
# 任务：尝试修改 test_ratio (测试集比例)，例如 0.2, 0.5, 0.9
# 思考：如果只有 10% 的数据用来学习，AI 还能学会吗？
test_ratio = 0.2 
# ---------------------------

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_ratio, random_state=42)

print(f"\n>>> 划分结果：")
print(f"   📚 训练集 (学习资料): {len(X_train)} 条")
print(f"   📝 测试集 (考试题目): {len(X_test)} 条")

## 第三阶段：模型是核心 —— 设计机器的大脑
我们要构建一个 **MLP (多层感知机)**。
*   **输入层**：接收 39 个数据点。
*   **隐藏层**：负责思考和特征提取（层数越多，神经元越多，越聪明，但也越慢）。
*   **输出层**：给出判断结果（站立/深蹲/半蹲）。

In [3]:
# === 【活动 2：构建神经网络】 ===
# 任务：自定义你的神经网络结构
# (10,)       代表：1个隐藏层，有10个神经元
# (64, 32)    代表：2个隐藏层，第1层64个，第2层32个神经元
# (100, 50, 20) 代表：3个隐藏层...
layer_sizes = (64, 32) 
# ----------------------------

# 初始化模型 (这时候它还是个只有架构的空壳，脑子是空白的)
model = MLPClassifier(hidden_layer_sizes=layer_sizes, 
                      solver='adam',      # 优化器 (负责调整参数)
                      random_state=42)

print(f"模型结构已构建！\n隐藏层结构: {layer_sizes}\n输入特征: {X.shape[1]}维 -> 输出类别: 3类")

NameError: name 'X' is not defined

## 第四阶段：综合优化 —— Loss 与 训练循环
机器的学习过程就是：**猜 -> 比(算Loss) -> 改** 的循环。
*   **Epoch (轮次)**：这个循环重复多少次？
*   **Loss (损失)**：错题罚分，越低越好。

In [4]:
# === 【活动 3：模型训练】 ===
# 任务：修改训练轮次，观察 Loss 曲线
# 建议尝试：10, 50, 200, 500
train_epochs = 200
# ---------------------------

print(f"=== 开始训练 (目标轮次: {train_epochs}) ===")

# 设置训练轮次
model.max_iter = train_epochs

# 开始训练 (Fit)
# 注意：每次运行这里，模型都会重置并从头开始训练
model.fit(X_train, y_train)

print("训练完成！")

# 绘制 Loss 曲线
plot_loss_curve(model.loss_curve_, train_epochs)

=== 开始训练 (目标轮次: 200) ===


NameError: name 'X_train' is not defined

## 第五阶段：成果验收
AI 裁判训练好了，是骡子是马，拉出来遛遛。

In [5]:
# 1. 总体考试成绩
score = model.score(X_test, y_test)
print(f"\n>>> 🏆 最终考试准确率: {score:.1%} (越高越好)")

# 2. 详细体检报告 (混淆矩阵)
# 它可以告诉你：AI 到底把“深蹲”误判成了“半蹲”，还是“站立”？
print("\n>>> 📊 判罚详情 (混淆矩阵):")
disp = ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, cmap=plt.cm.Blues)
plt.title("AI裁判判罚详情")
plt.show()

NameError: name 'X_test' is not defined

### 单个动作测试
抽取一道题，看看 AI 的具体分析过程。

In [6]:
print("=== 单个动作判罚演示 ===")

# 随机抽取1条测试数据
idx = 0 # 你可以修改这里抽取第几条数据
my_data = X_test[idx]
true_label = y_test[idx]

# 让模型预测
prediction = model.predict([my_data])[0]
probs = model.predict_proba([my_data])[0]
confidence = probs.max()

print(f"真实动作: 【 {true_label} 】")
print(f"AI判定:   【 {prediction} 】")
print(f"信心指数:   {confidence:.1%}")

if prediction == true_label:
    print("✅ 判罚正确！")
else:
    print("❌ 判罚错误！")
    
# 打印各个类别的概率
print("\nAI内心的概率分布:")
for label, prob in zip(model.classes_, probs):
    print(f"  - 是 {label} 的概率: {prob:.1%}")

=== 单个动作判罚演示 ===


NameError: name 'X_test' is not defined